In [ ]:
##Data Ingestion(LOAD)
from langchain_community.document_loaders import TextLoader
loader=TextLoader("speech.txt",encoding="utf-8")
text_documents=loader.load()
print(text_documents)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY']=os.getenv("OPENAI_API_KEY")

In [ ]:
# web based loader
from langchain_community.document_loaders import WebBaseLoader
import bs4

loader=WebBaseLoader(web_paths=("https://lilianweng.github.io/posts/2024-07-07-hallucination/",),
                     bs_kwargs=dict(parse_only=bs4.SoupStrainer(
                         class_=("post-title","post-content","post-header")
                     )))

text_documents=loader.load()


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
pdfloader=PyPDFLoader("llm_hallucinate.pdf")
docs=pdfloader.load()
docs

In [ ]:
##TRANSFORM
from langchain.text_splitter import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=250)
documents=text_splitter.split_documents(docs)
documents[:5]

In [41]:
##Vector Embedding & Vector Store
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
db=Chroma.from_documents(documents[:30],OllamaEmbeddings(model="nomic-embed-text"))

In [42]:
db

In [43]:
#Vector DB
query="Errors caused by pretraining"
result=db.similarity_search(query)
result[0].page_content

'While the analysis of pretraining covered errors more generally, our analysis of post-training focuses\non why overconfident hallucinations are generated rather than omitting information or expressing\nuncertainty such as IDK. We offer a socio-technical explanation for the persistence of hallucinations\nafter post-training and discuss how the field can suppress them.\n3'

In [44]:
##Advanced RAG system using Chains & retriever & prompts.
#Load ollama mistral LLM Model

from langchain_ollama import OllamaLLM
llm=OllamaLLM(model="mistral")
llm

OllamaLLM(model='mistral')

In [45]:
##Design Chat-Prompt template
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_template(
"""
Answer the following question strictly based only on the provided context.
Think step by step before providing detailed answer.
I will give you 5 star feedback if the user finds the answer helpful.
<context>
{context}
<context>
Question: {input}
"""
)

In [ ]:
##Chain Introduction
#chain constructor = create_stuff_documents_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
document_chain=create_stuff_documents_chain(llm,prompt)


In [47]:
#Retriever
retriever=db.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x0000027131EF9790>, search_kwargs={})

In [59]:
##need to combine doc chain & retriever from above to create RETRIEVAL CHAIN
from langchain.chains import create_retrieval_chain
retrieval_chain=create_retrieval_chain(retriever,document_chain)
response=retrieval_chain.invoke({"input":"Agnostic Learning(Kearns et al., 1994) addresses (a) by"})

In [60]:
print(response["answer"])

 Based on the provided context, Agnostic Learning as described by Kearns et al., 1994, addresses the problem of learning from errors where the underlying distribution that generates the data and the errors is unknown. The general sets of errors E are a combination of the errors and valid plausible strings V, with V being referred to as valid strings in the context. The models mentioned (DeepSeek-V3, chatgpt.com, DeepSeek-AI et al., 2025, Llama-4-Scout-17B-16E-Instruct) are not involved in searching the web, but they are used for accessing language models. However, the Agnostic Learning concept doesn't seem to be directly related to these models, as it is an older research work (1994) and the context primarily discusses more recent models.
